In [35]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet

ROOT = Path.cwd().parent

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

In [38]:
import numpy as np
import pandas as pd
from rapidfuzz import fuzz


def pair_exact_matches(df_left: pd.DataFrame, left_id_col: str,
                       df_right: pd.DataFrame, right_id_col: str) -> pd.DataFrame:
    """
    Exact matches via isbn_clean equality.
    Returns: id_left, id_right, label=1
    """
    L = df_left.dropna(subset=["isbn_clean"])[[left_id_col, "isbn_clean"]].rename(
        columns={left_id_col: "id_left", "isbn_clean": "isbn_l"}
    )
    R = df_right.dropna(subset=["isbn_clean"])[[right_id_col, "isbn_clean"]].rename(
        columns={right_id_col: "id_right", "isbn_clean": "isbn_r"}
    )

    exact = (
        L.merge(R, left_on="isbn_l", right_on="isbn_r", how="inner")
         .loc[:, ["id_left", "id_right"]]
         .drop_duplicates()
         .assign(label=1)
    )
    return exact


def pair_random_nonmatches(df_left: pd.DataFrame, left_id_col: str,
                           df_right: pd.DataFrame, right_id_col: str,
                           n: int, seed: int = 42) -> pd.DataFrame:
    """
    Randomly sample non-matching pairs (avoid equal isbn_clean when both present).
    Returns: id_left, id_right, label=0
    """
    rng = np.random.default_rng(seed)
    L = df_left[[left_id_col, "isbn_clean"]].to_numpy()
    R = df_right[[right_id_col, "isbn_clean"]].to_numpy()

    out, seen = [], set()
    BATCH = min(10000, max(1000, n * 5))

    while len(out) < n and len(seen) < len(L) * len(R):
        li = rng.integers(0, len(L), size=BATCH)
        ri = rng.integers(0, len(R), size=BATCH)
        for i, j in zip(li, ri):
            lid, lisbn = L[i]
            rid, risbn = R[j]
            if pd.notna(lisbn) and pd.notna(risbn) and str(lisbn) == str(risbn):
                continue
            key = (lid, rid)
            if key in seen:
                continue
            seen.add(key)
            out.append(key)
            if len(out) >= n:
                break

    return pd.DataFrame(out, columns=["id_left", "id_right"]).assign(label=0)


def _block_key(title: str, author: str) -> str:
    return " ".join(title.split()[:2] + author.split()[:1])


def _pair_sim(title_a: str, author_a: str, title_b: str, author_b: str) -> float:
    st = fuzz.token_set_ratio(title_a, title_b) / 100.0
    sa = fuzz.token_set_ratio(author_a, author_b) / 100.0
    return 0.7 * st + 0.3 * sa


def pair_corner_candidates(df_left: pd.DataFrame, left_id_col: str,
                           df_right: pd.DataFrame, right_id_col: str,
                           band_low: float = 0.70, band_high: float = 0.85) -> pd.DataFrame:
    """
    Surface borderline pairs based on title+author similarity within [band_low, band_high].
    Returns a review sheet.
    """
    L = df_left.copy()
    R = df_right.copy()

    L["blk"] = L.apply(lambda r: _block_key(r["title"], r["author"]), axis=1)
    R["blk"] = R.apply(lambda r: _block_key(r["title"], r["author"]), axis=1)

    C = L.merge(R, on="blk", suffixes=("_l", "_r"))
    if C.empty:
        return pd.DataFrame(columns=[
            "id_left", "id_right", "isbn_l", "isbn_r",
            "title_left", "author_left", "title_right", "author_right", "sim", "why", "label"
        ])

    C["sim"] = C.apply(lambda r: _pair_sim(r["title_l"], r["author_l"],
                                           r["title_r"], r["author_r"]), axis=1)

    C = C[(C["sim"] >= band_low) & (C["sim"] <= band_high)].copy()
    C["why"] = "text_boundary"

    review = (
    C.rename(columns={
        f"{left_id_col}_l": "id_left",
        f"{right_id_col}_r": "id_right",
        "isbn_clean_l": "isbn_l",
        "isbn_clean_r": "isbn_r",
        "title_l": "title_left",
        "author_l": "author_left",
        "title_r": "title_right",
        "author_r": "author_right",
    })
    .loc[:, [
        "id_left","id_right","isbn_l","isbn_r",
        "title_left","author_left","title_right","author_right","sim","why"
    ]]
    .drop_duplicates(subset=["id_left","id_right"])
    .assign(label=pd.NA)
    .sort_values("sim", ascending=False)
    .reset_index(drop=True)
)


    return review


In [40]:
# Example for Amazon ↔ Metabooks
pos_MA  = pair_exact_matches(amazon_sample, "id", metabooks_sample, "id")                 # label=1
corn_MA = pair_corner_candidates(amazon_sample, "id", metabooks_sample, "id", 0.7, 0.85) # to label manually later
neg_MA  = pair_random_nonmatches(amazon_sample, "id", metabooks_sample, "id", n=500, seed=42)  # label=0

# Example for Metabooks ↔ Goodreads
pos_MG  = pair_exact_matches(metabooks_sample, "id", goodreads_sample, "id")
corn_MG = pair_corner_candidates(metabooks_sample, "id", goodreads_sample, "id", 0.7, 0.85)
neg_MG  = pair_random_nonmatches(metabooks_sample, "id", goodreads_sample, "id", n=500, seed=42)


In [41]:
def compose_20_30_50(
    df_left: pd.DataFrame, left_id_col: str,
    df_right: pd.DataFrame, right_id_col: str,
    total_n: int,
    *,
    corner_band=(0.07, 0.85),
    seed: int = 42,
):
    """
    Build a labeled set with the classic 20/30/50 distribution:
      - 20% positives (exact ISBN matches)
      - 30% corners (text-similarity band; unlabeled -> to be labeled manually)
      - 50% negatives (random pairs avoiding equal ISBNs)
    Returns: (pos_df, corners_df, neg_df)
      pos_df: id_left, id_right, label=1
      corners_df: id_left, id_right, title_left, author_left, title_right, author_right, sim, why, label=NA
      neg_df: id_left, id_right, label=0
    """
    n_pos = int(round(total_n * 0.20))
    n_cor = int(round(total_n * 0.30))
    n_neg = total_n - n_pos - n_cor

    # 1) full pool of positives, then sample
    pos_pool = pair_exact_matches(df_left, left_id_col, df_right, right_id_col)
    pos = pos_pool.sample(n=min(n_pos, len(pos_pool)), random_state=seed) if len(pos_pool) else pos_pool

    # 2) full set of corners (banded), then sample
    low, high = corner_band
    corners_pool = pair_corner_candidates(df_left, left_id_col, df_right, right_id_col,
                                          band_low=low, band_high=high)
    corners = corners_pool.sample(n=min(n_cor, len(corners_pool)), random_state=seed) if len(corners_pool) else corners_pool

    # 3) sample negatives directly
    neg = pair_random_nonmatches(df_left, left_id_col, df_right, right_id_col, n=n_neg, seed=seed)

    return pos.reset_index(drop=True), corners.reset_index(drop=True), neg.reset_index(drop=True)


In [42]:
# Creating 1000 examples (Training- Validation- Test)

# Amazon ↔ Metabooks
pos_MA, corners_MA, neg_MA = compose_20_30_50(
    metabooks_sample, "id", amazon_sample, "id",
    total_n=1000,
    corner_band=(0.70, 0.85),
    seed=42
)
# Metabooks - Goodreads
pos_MG, corners_MG, neg_MG =compose_20_30_50(
    metabooks_sample,"id",goodreads_sample,"id",1000,    corner_band=(0.70, 0.85),   
    seed=42
)

In [24]:
import os
import pandas as pd

def export_triplets(df: pd.DataFrame, filename: str):
    """
    Write triplets with NO header:
        id_left,id_right,label
    Saves automatically inside ./sets/
    Assumes 'label' is 0/1 (no NaNs).
    """
    # Ensure ./sets folder exists
    os.makedirs("sets", exist_ok=True)
    path = os.path.join("sets", filename)

    required = {"id_left", "id_right", "label"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"export_triplets: missing columns {sorted(missing)}")

    # Enforce 0/1 integers (raise if NaN)
    if df["label"].isna().any():
        raise ValueError("export_triplets: 'label' contains NaN; only finalized 0/1 labels are allowed.")

    T = df[["id_left", "id_right", "label"]].copy()
    T["label"] = T["label"].astype(int)
    T.to_csv(path, index=False, header=False)
    print(f"✅ Triplets exported to: {path}")


def export_corner_review(
    corners_df: pd.DataFrame,
    filename: str,
    sample_n: int | None = None,
    seed: int = 42,
):
    """
    Export a review sheet for manual labeling with columns:
      id_left,id_right,title_left,author_left,title_right,author_right,sim,why,label
    - If sample_n is None or >= len(data), exports ALL rows.
    - Otherwise, exports a simple random sample (no stratification).
    Saves automatically inside ./sets/
    """
    import os
    os.makedirs("sets", exist_ok=True)
    path = os.path.join("sets", filename)

    cols = [
        "id_left","id_right","isbn_l","isbn_r",
        "title_left","author_left",
        "title_right","author_right",
        "sim","why","label",
    ]
    S = corners_df.copy()


    needed = {"id_left","id_right","isbn_l","isbn_r","title_left","author_left","title_right","author_right"}
    missing_needed = needed - set(S.columns)
    if missing_needed:
        raise KeyError(f"export_corner_review: missing columns {sorted(missing_needed)}")

    for c in ["sim","why","label"]:
        if c not in S.columns:
            S[c] = None

    if len(S) == 0:
        # nothing to export
        pd.DataFrame(columns=cols).to_csv(path, index=False)
        print(f"⚠️ No data to export — created empty file at {path}")
        return

    if sample_n is None or sample_n >= len(S):
        S[cols].to_csv(path, index=False)
    else:
        S.sample(n=sample_n, random_state=seed)[cols].to_csv(path, index=False)

    print(f"✅ Corner review sheet exported to: {path}")


In [47]:
def finalize_and_split(pos_df, corners_labeled_df, neg_df, *, test_frac=0.2, seed=42):
    """
    Merge positives + labeled corners + negatives, then stratified split (train/test only).
    Returns:
        train_df, test_df  (each with id_left, id_right, label)
    """
    # Merge all labeled pairs
    all_labeled = pd.concat(
        [
            pos_df[["id_left", "id_right", "label"]],
            corners_labeled_df[["id_left", "id_right", "label"]],
            neg_df[["id_left", "id_right", "label"]],
        ],
        ignore_index=True
    ).dropna(subset=["label"]).drop_duplicates()

    # Ensure binary integer labels
    all_labeled["label"] = all_labeled["label"].astype(int)

    # Shuffle once for randomness
    L = all_labeled.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    # Stratified split (preserve 0/1 balance)
    parts = {"train": [], "test": []}
    for y in (0, 1):
        grp = L[L["label"] == y]
        n = len(grp)
        n_test = int(round(n * test_frac))
        test_part = grp.iloc[:n_test]
        train_part = grp.iloc[n_test:]
        parts["train"].append(train_part)
        parts["test"].append(test_part)

    train = pd.concat(parts["train"], ignore_index=True)
    test  = pd.concat(parts["test"],  ignore_index=True)
    return train, test

In [49]:
# Manually label the corners and then split all examples into train/val/test
#labeled_corners_MG = pd.read_csv("sets/corners_MG_review.csv")
train_MG,test_MG =finalize_and_split(pos_MG,neg_MG,corners_MG,test_frac=0.2,seed=42)

In [50]:
# Doing the same for other pair
#labeled_corners_MA=pd.read_csv("sets/corners_MA_review.csv")
train_MA, test_MA =finalize_and_split(pos_MA,neg_MA,corners_MA,test_frac=0.2,seed=42)

In [51]:
display(train_MG.label.describe())
display(test_MG.label.describe())
# Stratified split

count    800.00000
mean       0.20000
std        0.40025
min        0.00000
25%        0.00000
50%        0.00000
75%        0.00000
max        1.00000
Name: label, dtype: float64

count    200.000000
mean       0.200000
std        0.401004
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: label, dtype: float64

In [54]:
train_MG.to_csv(ROOT/"ml-datasets/train_MG.csv",index=False)
test_MG.to_csv(ROOT/"ml-datasets/test_MG.csv",index=False)
train_MA.to_csv(ROOT/"ml-datasets/train_MA.csv",index=False)
test_MA.to_csv(ROOT/"ml-datasets/test_MA.csv",index=False)